[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/relativity/kerr/kerr_gpu.ipynb)

# Light Past a Rotating Black Hole, Traced on the GPU

This example traces camera light rays backward on the GPU from an observer above a rotating Kerr black hole through a thin, glowing accretion disk to the background sky or event horizon. Gravity bends the disk's light around the hole's shadow, lifts the far side of the disk above and below the shadow, shifts approaching gas blue and receding gas red, and turns linear polarization along each ray. Running one ray per GPU invocation traces each frame from scratch as the gas orbits and a linear polarizer turns in front of the camera.

In numga's WGSL dialect, the position gauge field `Antivector + dyad(geometry)` is built and applied as a typed linear map `gauge(geometry)(heading)`, while `geometry.null_gradient.adjugate()(ray.momentum)` pulls the momentum back through the null direction's derivative without coordinate components. Along each ray, the rotation gauge bivector transports the transverse camera screen by `-rotation(geometry, heading).commutator(ray.screen)`, and `analyzer & disk_coherency(crossing)(analyzer)` reads the transmitted share at each disk crossing. Every nullary or unary extensor compiles to a WGSL struct of its nonzero coefficients with symbolically generated operator bodies. Companion notebooks develop the gauge-field physics on the CPU in [kerr.ipynb](kerr.ipynb), the metric and Christoffel formulation in [kerr_metric.ipynb](kerr_metric.ipynb), and their side-by-side equivalence in [kerr_comparison.ipynb](kerr_comparison.ipynb).

In tensor-index notation, a handwritten GPU ray tracer unpacks the Kerr–Schild metric $g_{\mu\nu} = \eta_{\mu\nu} - 2 H l_\mu l_\nu$ and inverse $g^{\mu\nu} = \eta^{\mu\nu} + 2 H l^\mu l^\nu$ into four-component vectors, differentiates $H(x, p) = \tfrac{1}{2} g^{\mu\nu} p_\mu p_\nu$ through explicit index sums $\dot{x}^\mu = g^{\mu\nu} p_\nu$ and $\dot{p}_\alpha = -\partial_\alpha H\,(l^\mu p_\mu)^2 - 2 H\,(l^\mu p_\mu)\,p_\nu \partial_\alpha l^\nu$, and transports two transverse polarization four-vectors through the sixty-four Christoffel components $\Gamma^\mu_{\alpha\beta} = \tfrac{1}{2} g^{\mu\sigma}(\partial_\alpha g_{\sigma\beta} + \partial_\beta g_{\sigma\alpha} - \partial_\sigma g_{\alpha\beta})$ at every Runge–Kutta stage.

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import Algebra, NumpyContext
from numga.algebras import STA as ga
from numga.wgsl import Shader, supersampled
from examples.util.animation import save_figure, show_animation
from examples.relativity.kerr import render, scenarios

mv = NumpyContext(ga).multivector
Vector = ga.gatype.vector()
Bivector = ga.gatype.bivector()
Antivector = ga.gatype.antivector()
# The hole's spin per unit mass: intrinsic, the patch of the plane it turns in, a plane through time.
Spin = Bivector.derive.within(Algebra("x+y+z+")).derive.dual
Rotor = ga.gatype.rotor()
# The camera's slices x and y, which read screen directions: vectors, extrinsic.
ScreenSlice = Vector.derive.within(Algebra("x+y+"))
# Directions on the camera's screen, along its own x and y: the antivectors that hold both t and z.
Screen = ScreenSlice.derive.dual
ScreenFrame = ga.gatype((Antivector, Screen))                # Antivector- <- Screen-
# The light's polarization covariance, from the slices that read the screen to screen directions.
Coherency = ga.gatype((Screen, ScreenSlice))                 # Screen- <- ScreenSlice+
# The position gauge field: from a direction of the local frame to one of the Kerr geometry.
GaugeField = ga.gatype((Antivector, Antivector))             # Antivector- <- Antivector-
types = dict(Vector=Vector, Bivector=Bivector, Antivector=Antivector, Spin=Spin, Rotor=Rotor, Screen=Screen,
             ScreenSlice=ScreenSlice, ScreenFrame=ScreenFrame, Coherency=Coherency, GaugeField=GaugeField)
# The camera's own axes in its rest frame, before it is turned: it looks along -z, with its right +y and its up -x.
camera_forward = -mv.z.dual()                                # [] Antivector-
camera_right = mv.y.dual()                                   # [] Antivector-
camera_up = -mv.x.dual()                                     # [] Antivector-
constants = dict(camera_forward=camera_forward, camera_right=camera_right, camera_up=camera_up)

## Kerr–Schild geometry in WGSL

At each event the geometry reduces to a scalar profile falling off with the oblate radius and an ingoing null direction along which light falls into the hole. Together with their spatial derivatives they form the position gauge field `Antivector + dyad(geometry)` and its exact inverse `Antivector - dyad(geometry)`. Constant maps known when the shader compiles fold directly into the generated WGSL functions, so `space(position)` projects onto the lab's spatial slice with no extra runtime storage.

In [ ]:
geometry = "\n".join([
    "const pi = 3.141592653589793;",
    "",
    "// One unit along the lab's time, and the part of a direction within the lab's slice of space.",
    "const time = -mv.t.dual();",
    "const space = Antivector - time * (mv.t & Antivector);",
    "",
    "var<uniform> spin: Spin;",
    "var<uniform> mass: f32;",
    "",
    "struct Field {",
    "    radius: f32,",
    "    profile: f32,",
    "    null: Antivector,",
    "    profile_gradient: Vector,",
    "    null_gradient: GaugeField,",
    "    null_curl: Bivector,",
    "}",
    "",
    "fn field(position: Antivector) -> Field {",
    "    let spatial = space(position);",
    "    // The plane the hole turns in, whose dual is its spin; the volume the position spans with it, and the",
    "    // position's height off it.",
    "    let turning = spin.dual_inverse();",
    "    let volume = spatial.anticommutator(turning);",
    "    let height = volume ^ turning;",
    "    let spin_squared = spin.squared();",
    "    let offset = spatial.squared() - spin_squared;",
    "    let discriminant = (offset.squared() - 4 * volume.squared()).square_root();",
    "    let radius_squared = (offset + discriminant) / 2;",
    "    let radius = radius_squared.square_root();",
    "    // Paired with a small step, how far the step changes the oblate radius.",
    "    let radius_gradient = (radius_squared * spatial - height).dual() / (radius * discriminant);",
    "    // The hole drags the outgoing direction against its turning.",
    "    let direction = (radius * spatial - spatial.commutator(turning) - height / radius) / (radius_squared + spin_squared);",
    "    let lever = spatial - 2 * radius * direction + height / radius_squared;",
    "    let direction_gradient = (",
    "        lever * (radius_gradient & Antivector)",
    "        + radius * space - space.commutator(turning)",
    "        - (space.anticommutator(turning) ^ turning) / radius",
    "    ) / (radius_squared + spin_squared);",
    "    let denominator = radius_squared.squared() - volume.squared();",
    "    let profile = mass * radius**3 / denominator;",
    "    let profile_gradient = profile * (3 * radius_gradient / radius",
    "        - (4 * radius**3 * radius_gradient - 2 * height.dual()) / denominator);",
    "    let null_curl = ((radius_gradient ^ lever.dual()) - 2 * turning) / (radius_squared + spin_squared);",
    "    return Field(radius, profile, time - direction, profile_gradient, -direction_gradient, null_curl);",
    "}",
    "",
    "fn null_plane(geometry: Field) -> Vector {",
    "    return -geometry.null.dual();",
    "}",
    "",
    "// The position gauge field: the identity plus the profile times the null dyad, which squares to zero.",
    "fn dyad(geometry: Field) -> GaugeField {",
    "    return geometry.profile * geometry.null * (null_plane(geometry) & Antivector);",
    "}",
    "",
    "fn gauge(geometry: Field) -> GaugeField {",
    "    return Antivector + dyad(geometry);",
    "}",
    "",
    "fn gauge_inverse(geometry: Field) -> GaugeField {",
    "    return Antivector - dyad(geometry);",
    "}",
    "",
    "// A momentum's direction in the local frame: raised flat and carried through the gauge field.",
    "fn local(geometry: Field, momentum: Vector) -> Antivector {",
    "    return gauge(geometry)(-momentum.dual());",
    "}",
    "",
    "fn proper_squared(geometry: Field, direction: Antivector) -> f32 {",
    "    return -gauge_inverse(geometry)(direction).squared();",
    "}",
    "",
    "// The gradient of a momentum's pairing with the null direction: the momentum pulled back.",
    "fn overlap_gradient(geometry: Field, momentum: Vector) -> Vector {",
    "    return geometry.null_gradient.adjugate()(momentum);",
    "}",
    "",
])

## Ray motion and screen transport

Each ray carries its spacetime event, its covector momentum, and the camera's transverse screen frame in a single WGSL struct. Because structs whose fields are algebra extensors add and scale field by field, one fourth-order Runge–Kutta step advances the entire ray state in a single expression `ray + (first + 2 * second + 2 * third + fourth) * (step / 6)`. Scaling by decreasing coordinate time steps all rays backward from the camera toward the disk, horizon, and sky.

In [ ]:
rays = "\n".join([
    "// A ray: its event, its momentum, and the camera's screen carried along it.",
    "struct Ray {",
    "    position: Antivector,",
    "    momentum: Vector,",
    "    screen: ScreenFrame,",
    "}",
    "",
    "// The rotation gauge field: the plane the local frame turns in per unit step along a direction.",
    "fn rotation(geometry: Field, direction: Antivector) -> Bivector {",
    "    let null = null_plane(geometry);",
    "    let along = null & direction;",
    "    let turning = geometry.profile_gradient * along + geometry.profile * overlap_gradient(geometry, -direction.dual());",
    "    return -(turning ^ null) - geometry.profile * geometry.null_curl * along;",
    "}",
    "",
    "// Hamiltonian motion and parallel transport of the screen, per unit of decreasing coordinate time.",
    "fn rates(ray: Ray) -> Ray {",
    "    let geometry = field(ray.position);",
    "    let overlap = ray.momentum & geometry.null;",
    "    let heading = local(geometry, ray.momentum);",
    "    let tangent = gauge(geometry)(heading);",
    "    let gradient = geometry.profile_gradient * overlap.squared()",
    "        + 2 * geometry.profile * overlap * overlap_gradient(geometry, ray.momentum);",
    "    let screen_rate = -rotation(geometry, heading).commutator(ray.screen);",
    "    return Ray(tangent, -gradient, screen_rate) / -(mv.t & tangent);",
    "}",
    "",
    "fn runge_kutta(ray: Ray, step: f32) -> Ray {",
    "    let first = rates(ray);",
    "    let second = rates(ray + first * (step / 2));",
    "    let third = rates(ray + second * (step / 2));",
    "    let fourth = rates(ray + third * step);",
    "    return ray + (first + 2 * second + 2 * third + fourth) * (step / 6);",
    "}",
    "",
])

## Thin accretion disk

Equatorial gas orbits with the hole down to the innermost stable circular orbit of [Bardeen, Press and Teukolsky (1972)](https://articles.adsabs.harvard.edu/pdf/1972ApJ...178..347B) and radiates with the relativistic thin-disk temperature profile of [Page and Thorne (1974)](https://articles.adsabs.harvard.edu/pdf/1974ApJ...191..499P). At each equatorial crossing, the disk absorbs according to the slant optical depth in the orbiting gas frame and emits blackbody light shifted in temperature by `(crossing.momentum & at_rest()) / (crossing.momentum & gas)`. Turning each crossing event backward along its orbit via `(orbit * (angle / 2)).exp() >> position` evaluates the procedural clump waves where the gas sat when the light was emitted, while `disk_coherency` forms the partly polarized emission covariance across the transported screen.

In [ ]:
disk = "\n".join([
    "// The disk lies in the plane `orbit`, its height measured by `normal`.",
    "const orbit = mv.xy;",
    "const normal = mv.z;",
    "",
    "var<uniform> outer_radius: f32;",
    "var<uniform> inner_depth: f32;",
    "var<uniform> polarization_degree: f32;",
    "var<uniform> clump_contrast: f32;",
    "var<uniform> elapsed: f32;",
    "var<uniform> flow_time: f32;",
    "var<uniform> fade: f32;",
    "var<uniform> hottest: f32;",
    "// The disk's temperature scale, (accretion rate c^6 / (sigma G^2 M^2))^(1/4), and blackbody light up to the hottest.",
    "var<uniform> temperature_scale: f32;",
    "var blackbodies: texture_2d<f32>;",
    "var filtered: sampler;",
    "// One of the disk's random waves: its whole turns around the disk, waves per unit radius across it, and phase.",
    "struct Mode {",
    "    turns: f32,",
    "    wavenumber: f32,",
    "    phase: f32,",
    "}",
    "",
    "var<storage, read> modes: array<Mode>;",
    "",
    "// The radius of the innermost stable circular orbit in the disk's plane, the spin counted along the orbit, negative",
    "// for gas orbiting against the hole: gas orbiting with the hole reaches inward of the radius without spin, gas",
    "// orbiting against it outward (Bardeen, Press and Teukolsky, 1972).",
    "fn inner_radius() -> f32 {",
    "    let ratio = (orbit & spin) / mass;",
    "    let first = 1 + pow(1 - ratio * ratio, 1.0 / 3) * (pow(1 + ratio, 1.0 / 3) + pow(1 - ratio, 1.0 / 3));",
    "    let second = sqrt(3 * ratio * ratio + first * first);",
    "    let reach = sqrt((3 - first) * (3 + first + 2 * second));",
    "    return mass * (3 + second - select(reach, -reach, ratio < 0));",
    "}",
    "",
    "fn orbital_speed(radius: f32) -> f32 {",
    "    let alignment = orbit & spin;",
    "    return mass.square_root() / (radius * radius.square_root() + alignment * mass.square_root());",
    "}",
    "",
    "// The unit velocity of the gas orbiting at an event: a turned point moves by its commutator with the plane.",
    "fn orbiting_gas(position: Antivector, radius: f32) -> Antivector {",
    "    let flow = time + orbital_speed(radius) * space(position).commutator(orbit);",
    "    return flow / proper_squared(field(position), flow).square_root();",
    "}",
    "",
    "// The share of a ray's light the disk absorbs where it crosses: its depth over the slant the gas sees.",
    "fn disk_opacity(crossing: Ray, radius: f32) -> f32 {",
    "    let inner = inner_radius();",
    "    let fraction = (radius - inner) / (outer_radius - inner);",
    "    let depth = inner_depth * (1 - fraction) * (1 - fraction);",
    "    let gas = orbiting_gas(crossing.position, radius);",
    "    let slant = abs((crossing.momentum & gas) / (crossing.momentum & normal.dual()));",
    "    return 1 - exp(-depth * slant);",
    "}",
    "",
    "// The disk's random waves where the gas seen at `position` was a time `before` ago. The clock starts when light leaves",
    "// the hole's centre for the camera at its `distance`, so that moving the camera does not turn the disk.",
    "fn waves(position: Antivector, radius: f32, before: f32) -> f32 {",
    "    let angle = orbital_speed(radius) * (elapsed - before + (mv.t & position) + distance);",
    "    let carried = (orbit * (angle / 2)).exp() >> position;",
    "    let around = atan2(mv.y & carried, mv.x & carried);",
    "    var total = 0.0;",
    "    for (var index = 0u; index < arrayLength(&modes); index++) {",
    "        let mode = modes[index];",
    "        total += sin(mode.turns * around + mode.wavenumber * radius + mode.phase);",
    "    }",
    "    return total / sqrt(f32(arrayLength(&modes)) / 2);",
    "}",
    "",
    "// A relativistic thin disk's temperature at an oblate radius, from the flux of Page and Thorne (1974): zero at the",
    "// innermost stable orbit, falling as the radius to the power -3/4 far out.",
    "fn rest_temperature(radius: f32) -> f32 {",
    "    let ratio = (orbit & spin) / mass;",
    "    let root_radius = sqrt(radius / mass);",
    "    let edge = sqrt(inner_radius() / mass);",
    "    // The roots of the cubic `root_radius**3 - 3 * root_radius + 2 * ratio`, and the weight of each in the torque.",
    "    let roots = 2 * cos((acos(ratio) + vec3<f32>(-pi, pi, 3 * pi)) / 3);",
    "    let weights = (roots - ratio) * (roots - ratio) / (roots * (roots * roots - 1));",
    "    // The torque the disk carries, zero at the innermost stable orbit.",
    "    let terms = weights * log((root_radius - roots) / (edge - roots));",
    "    let torque = max(root_radius - edge - 1.5 * ratio * log(root_radius / edge) - (terms.x + terms.y + terms.z), 0.0);",
    "    let cubic = root_radius * root_radius * root_radius - 3 * root_radius + 2 * ratio;",
    "    let flux = 3 / (8 * pi) * torque / (root_radius * root_radius * root_radius * root_radius * cubic);",
    "    return temperature_scale * sqrt(sqrt(flux));",
    "}",
    "",
    "// Linear light the disk emits towards the camera: a blackbody at the shifted temperature, textured.",
    "fn disk_light(crossing: Ray, radius: f32) -> vec3<f32> {",
    "    let gas = orbiting_gas(crossing.position, radius);",
    "    // A blackbody seen at a shifted frequency is a blackbody at the shifted temperature.",
    "    let shift = (crossing.momentum & at_rest()) / (crossing.momentum & gas);",
    "    // The table's entries sit at its texel centres, from zero to the hottest temperature.",
    "    let entries = f32(textureDimensions(blackbodies).x);",
    "    let entry = clamp(shift * rest_temperature(radius) / hottest, 0.0, 1.0) * (entries - 1) + 0.5;",
    "    let colour = textureSampleLevel(blackbodies, filtered, vec2<f32>(entry / entries, 0.5), 0.0).rgb;",
    "    let rings = 0.90 + 0.045 * sin(11 * radius) + 0.025 * sin(19 * radius);",
    "    let clumps = cos(fade) * waves(crossing.position, radius, 0.0) + sin(fade) * waves(crossing.position, radius, flow_time);",
    "    return colour * rings * exp(clump_contrast * clumps);",
    "}",
    "",
    "// The disk's emission, a covariance from the camera's slices to screen directions: partly polarized square to the",
    "// magnetic field circling the normal.",
    "fn disk_coherency(crossing: Ray) -> Coherency {",
    "    let heading = local(field(crossing.position), crossing.momentum);",
    "    let direction = space(heading) / (mv.t & heading);",
    "    let meridian = space(crossing.position).dual() ^ normal;",
    "    let polarization_axis = meridian.commutator(direction) / meridian.norm();",
    "    let polarization = crossing.screen.adjoint()(polarization_axis);",
    "    let polarized_fraction = polarization_degree * polarization.squared();",
    "    // Unpolarized light is round, where the metric enters; the polarized part weights the polarization by each",
    "    // slice's reading of it.",
    "    return (1 - polarized_fraction) / 2 * ScreenSlice.dual() + polarization_degree * polarization * (ScreenSlice & polarization);",
    "}",
    "",
])

## Camera rays and static view

Each invocation traces a ray backward from the static camera until it escapes to the celestial sphere or crosses inside the capture radius, accumulating direct and lensed disk emission, analyzer-transmitted emission, and background starlight. Passing `supersampled()` averages a rotated subpixel grid of `Sample` structs per pixel before tone mapping, while coordinates are extracted only at texture lookups through basis pairings. Compiling the WGSL shader and tracing the supersampled view takes a few seconds.

In [ ]:
camera = "\n".join([
    "var<uniform> azimuth: f32;",
    "var<uniform> inclination: f32;",
    "var<uniform> distance: f32;",
    "var<uniform> analyzer: ScreenSlice;",
    "var<uniform> half_view: f32;",
    "var<uniform> step_fraction: f32;",
    "var<uniform> n_steps: u32;",
    "var<uniform> escape_radius: f32;",
    "var stars: texture_2d<f32>;",
    "",
    "// What a ray brings back: the disk's light, the share of it an analyzer passes, and the starlight.",
    "struct Sample {",
    "    light: vec3<f32>,",
    "    analyzed: vec3<f32>,",
    "    sky: vec3<f32>,",
    "}",
    "",
    "// The camera's orientation: turned by its inclination from above the hole, then by its azimuth around it.",
    "fn orientation() -> Rotor {",
    "    return (mv.xy * (-azimuth / 2)).exp() * (mv.zx * (-inclination / 2)).exp();",
    "}",
    "",
    "// The camera's eye, at its distance from the hole along its own backward direction.",
    "fn eye() -> Antivector {",
    "    return (orientation() >> mv.z.dual()) * distance;",
    "}",
    "",
    "// The camera at rest: a unit along the lab's time measured in proper time at the eye.",
    "fn at_rest() -> Antivector {",
    "    return time / proper_squared(field(eye()), time).square_root();",
    "}",
    "",
    "// The radius inside every photon orbit, where no ray turns back out.",
    "fn capture_radius() -> f32 {",
    "    return 1.05 * (mass + sqrt(mass * mass - spin.squared()));",
    "}",
    "",
    "// The static camera's rest frame carried into the local Kerr–Schild frame.",
    "fn camera_boost(geometry: Field) -> Rotor {",
    "    let carried_time = gauge_inverse(geometry)(time);",
    "    let observer = carried_time / (-carried_time.squared()).square_root();",
    "    return (1 - observer * time).normalized();",
    "}",
    "",
    "// The camera's own right and up across a ray, as a map from the screen's two axes.",
    "fn camera_screen(position: Antivector, momentum: Vector) -> ScreenFrame {",
    "    let geometry = field(position);",
    "    let boost = camera_boost(geometry);",
    "    let heading = boost.reverse() >> local(geometry, momentum);",
    "    let direction = space(heading) / -(mv.t & heading);",
    "    // The turn carrying the camera's forward onto the direction it looks to see the ray: the reflection in the",
    "    // forward, then in the direction halfway between the two. It turns the camera's right and up with it.",
    "    let forward = orientation() >> camera_forward;",
    "    let halfway = direction + forward;",
    "    let turn = halfway * forward / halfway.squared().square_root();",
    "    return (boost * turn * orientation()) >> (camera_right * (mv.x & Screen) + camera_up * (mv.y & Screen));",
    "}",
    "",
    "// The starlight a ray escaping along a direction brings: the sky's image at its longitude and latitude.",
    "fn starlight(direction: Antivector) -> vec3<f32> {",
    "    let longitude = atan2(mv.y & direction, mv.x & direction);",
    "    let latitude = asin((mv.z & direction) / space(direction).squared().square_root());",
    "    let place = vec2<f32>(longitude / (2 * pi) + 0.5, 0.5 - latitude / pi);",
    "    return textureSampleLevel(stars, filtered, place, 0.0).rgb;",
    "}",
    "",
    "// The light along the ray through a place on the image, followed back from the camera.",
    "fn sample(place: vec2<f32>) -> Sample {",
    "    let across = screen(place) * half_view;",
    "    // Past-directed light from the pixel, looking along the camera's -z.",
    "    var direction = orientation() >> (camera_forward + camera_right * across.x + camera_up * across.y);",
    "    direction = direction / direction.squared().square_root();",
    "    let start = eye();",
    "    let geometry = field(start);",
    "    let heading = camera_boost(geometry) >> (time - direction);",
    "    let future_momentum = -gauge_inverse(geometry)(heading).dual();",
    "    let momentum = -future_momentum / (future_momentum & time);",
    "    var ray = Ray(start, momentum, camera_screen(start, momentum));",
    "    let inner = inner_radius();",
    "    let capture = capture_radius();",
    "",
    "    var radius = field(ray.position).radius;",
    "    var remaining = 1.0;",
    "    var disk = vec3<f32>(0.0);",
    "    var through = vec3<f32>(0.0);",
    "    var stars_seen = vec3<f32>(0.0);",
    "    for (var step = 0u; step < n_steps; step++) {",
    "        let end = runge_kutta(ray, step_fraction * radius);",
    "        let end_radius = field(end.position).radius;",
    "        // Where the step crosses the disk's plane, between its edges, the disk takes a share of the light.",
    "        if ((normal & ray.position) * (normal & end.position) <= 0) {",
    "            let crossing = ray + (end - ray) * (-(normal & ray.position) / (normal & (end.position - ray.position)));",
    "            let crossing_radius = field(crossing.position).radius;",
    "            if (crossing_radius >= inner && crossing_radius <= outer_radius) {",
    "                let weight = remaining * disk_opacity(crossing, crossing_radius);",
    "                let emitted = weight * disk_light(crossing, crossing_radius);",
    "                disk += emitted;",
    "                through += emitted * (analyzer & disk_coherency(crossing)(analyzer));",
    "                remaining -= weight;",
    "            }",
    "        }",
    "        if (end_radius > escape_radius) {",
    "            let geometry = field(end.position);",
    "            stars_seen = remaining * starlight(gauge(geometry)(local(geometry, end.momentum)));",
    "            break;",
    "        }",
    "        if (end_radius < capture) {",
    "            break;",
    "        }",
    "        ray = end;",
    "        radius = end_radius;",
    "    }",
    "    return Sample(disk, through, stars_seen);",
    "}",
    "",
])

In [ ]:
n_table_entries = 256
hottest = 3 * scenarios.disk_temperature_scale
spin = mv.xy.dual() * scenarios.camera_spin                                      # [] Spin-
image_shape = (scenarios.n_image_height, scenarios.n_image_width)
values = dict(
    spin=spin, mass=scenarios.mass, azimuth=0.0, inclination=scenarios.camera_inclination,
    distance=scenarios.camera_distance,
    outer_radius=scenarios.disk_outer_radius, inner_depth=scenarios.inner_depth,
    polarization_degree=scenarios.polarization_degree, clump_contrast=render.clump_contrast,
    elapsed=0.0, flow_time=scenarios.flow_time, fade=0.0, analyzer=mv.x,
    hottest=hottest, temperature_scale=scenarios.disk_temperature_scale,
    blackbodies=render.blackbody_table(hottest, n_table_entries), modes=render.wave_modes(),
    half_view=scenarios.half_view, step_fraction=scenarios.camera_step_fraction,
    n_steps=scenarios.n_camera_steps, escape_radius=scenarios.escape_radius,
    stars=render.star_texture(scenarios.n_sky_width, scenarios.n_sky_height, scenarios.sky_seed))

tracer = Shader(geometry + rays + disk + camera + supersampled(), ga, types, constants)
still = tracer(image_shape, **values)
save_figure(render.frame_figure(render.tone_mapped_frame(still["image"]["light"], still["image"]["sky"], scenarios.exposure_temperature)), "kerr_gpu_camera");

Gravitational lensing lifts the far half of the equatorial disk into a bright arch above the central shadow and a fainter secondary image underneath it, framing a narrow photon ring around the capture boundary. Relativistic beaming and Doppler shift brighten and shift the approaching gas on the left toward blue-white while dimming and reddening the receding gas on the right.

## Orbiting disk flow

Stepping the elapsed coordinate time traces every frame anew while cross-fading the retrodicted clump wave field across two orbital periods so the loop closes seamlessly. Tracing all frames of the orbital loop takes several seconds.

In [ ]:
n_flow_frames = scenarios.n_flow_frames


def disk_flow_samples(tracer: Shader, values: dict) -> Iterator[dict]:
    """Yields GPU-traced pixel samples across one orbital loop of the accretion disk."""
    for index, elapsed in enumerate(np.linspace(0, scenarios.flow_time, n_flow_frames, endpoint=False)):
        yield tracer(image_shape, **values | dict(elapsed=elapsed, fade=np.pi / 2 * index / n_flow_frames))


show_animation((render.tone_mapped_frame(traced["image"]["light"], traced["image"]["sky"], scenarios.exposure_temperature)
                for traced in disk_flow_samples(tracer, values)),
               "kerr_gpu_disk", scenarios.frame_duration_ms, n_frames=n_flow_frames)

Keplerian shear makes the inner disk complete several turns while the outer rings advance through a fraction of an orbit, winding bright clumps into trailing spiral arcs. Light-travel delay across the curved paths shears the lensed upper arch relative to the foreground disk.

## Rotating linear polarizer

Turning a linear analyzer slice in front of the camera evaluates `analyzer & disk_coherency(crossing)(analyzer)` at every disk crossing while doubling the disk exposure to compensate for the half of unpolarized emission blocked by an ideal polarizer.

In [ ]:
n_polarizer_frames = scenarios.n_polarizer_frames
analyzer_rotors = (mv.xy * (-scenarios.polarizer_angles / 2)).exp()              # [n_polarizer_frames] Rotor+
analyzers = analyzer_rotors >> mv.x                                              # [n_polarizer_frames] ScreenSlice+


def polarizer_samples(tracer: Shader, values: dict) -> Iterator[dict]:
    """Yields GPU-traced pixel samples across one half-turn of the linear polarizer."""
    for analyzer in analyzers:
        yield tracer(image_shape, **values | dict(analyzer=analyzer))


show_animation((render.tone_mapped_frame(2 * traced["image"]["analyzed"], traced["image"]["sky"], scenarios.exposure_temperature)
                for traced in polarizer_samples(tracer, values)),
               "kerr_gpu_polarizer", scenarios.frame_duration_ms, n_frames=n_polarizer_frames)

As the analyzer turns through half a revolution, a dark extinction band sweeps smoothly around both the direct disk and the lensed upper arch wherever the transported polarization lies perpendicular to the analyzer axis, while unpolarized background starlight stays constant.

In [ ]:
# checks
assert ((spin.squared() - scenarios.camera_spin**2).abs() < 1e-12).all()
assert ((analyzers.squared() - 1).abs() < 1e-12).all()
for traced in still["image"].values():
    assert np.all(np.isfinite(traced)) and np.all(traced >= 0)
assert np.all(still["image"]["analyzed"] <= still["image"]["light"] + 1e-5)